# AeroSense: Cough-Based Respiratory Sound Classification (Coswara)
**Task:** classify a cough recording into 3 classes: `healthy`, `covid_positive`, `recovered`
**Pipeline:** metadata -> audio -> log-Mel spectrograms -> patient-level split -> CNN -> evaluation

Run the cells from top to bottom. Each cell prints what it is doing, so you can see where a problem happens.

## 1. Setup & Environment Check

In [ ]:
# --- 1.1 Imports -------------------------------------------------------------
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import tensorflow as tf

warnings.filterwarnings("ignore")   # keep the notebook output clean
print("TensorFlow version:", tf.__version__)
print("Librosa version   :", librosa.__version__)
print("GPU available     :", len(tf.config.list_physical_devices("GPU")) > 0)

In [ ]:
# --- 1.2 Mount Google Drive --------------------------------------------------
# In Colab this asks you to authorise access to your Drive.
# If you run this somewhere else (e.g. your laptop), it simply skips mounting.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
except ImportError:
    print("Not running in Colab -> skipping Drive mount (using local paths).")

In [ ]:
# --- 1.3 Define dataset paths ------------------------------------------------
# Nothing needs to be downloaded by hand: the next cells fetch Coswara from its official
# GitHub repository (https://github.com/iiscleap/Coswara-Data) and keep only the cough recordings.
import re
import shlex
import shutil
import subprocess
import urllib.request

# Everything is stored in Google Drive (if available) so you only download once.
if os.path.isdir("/content/drive/MyDrive"):
    BASE_DIR = "/content/drive/MyDrive/AeroSense"
else:
    BASE_DIR = "./AeroSense"          # fallback when not using Colab/Drive

GITHUB_REPO_URL = "https://github.com/iiscleap/Coswara-Data.git"
METADATA_URL    = "https://raw.githubusercontent.com/iiscleap/Coswara-Data/master/combined_data.csv"

DATASET_ROOT = os.path.join(BASE_DIR, "Coswara")                 # permanent copy (Drive)
METADATA_CSV = os.path.join(DATASET_ROOT, "combined_data.csv")   # metadata
AUDIO_ROOT   = os.path.join(DATASET_ROOT, "cough_audio")         # extracted cough wav files
OUTPUT_DIR   = os.path.join(BASE_DIR, "outputs")                 # models / plots
CLONE_DIR    = "/content/Coswara-Data" if os.path.isdir("/content") else "./Coswara-Data_tmp"  # temporary

# Optional: only download the first N date-folders (useful for a quick test run).
# None = download everything (recommended for the final results).
MAX_DATE_FOLDERS = None

for folder in [DATASET_ROOT, AUDIO_ROOT, OUTPUT_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Dataset will be stored in:", DATASET_ROOT)
print("Outputs will be saved to :", OUTPUT_DIR)

In [ ]:
# --- 1.4 Download Coswara from GitHub (runs only once) -----------------------
# Coswara has no REST API; its official distribution is a GitHub repository where each
# date-folder holds a compressed archive of audio files. We:
#   A) download the metadata CSV,
#   B) clone the repository (temporary, on Colab's local disk),
#   C) extract ONLY the cough recordings from every archive into Google Drive,
#   D) delete the temporary clone to free disk space.
# Expect roughly 10-30 minutes depending on GitHub/Colab speed. Re-running this cell is safe:
# it skips everything that is already done.
from tqdm.auto import tqdm

DONE_MARKER = os.path.join(AUDIO_ROOT, ".download_complete")

# --- A) metadata CSV ---
if not os.path.exists(METADATA_CSV):
    print("Downloading metadata CSV ...")
    urllib.request.urlretrieve(METADATA_URL, METADATA_CSV)
print("Metadata CSV ready:", METADATA_CSV)

# --- B-D) audio ---
if os.path.exists(DONE_MARKER):
    print("Cough audio already downloaded - skipping.")
else:
    # B) shallow clone (only the latest snapshot, no git history)
    if not os.path.isdir(CLONE_DIR):
        print("Cloning the Coswara repository (this is the slow step) ...")
        result = subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, CLONE_DIR],
                                capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError("git clone failed:\n" + result.stderr[-500:] +
                               "\nCheck your internet connection and run this cell again.")

    # date folders look like 20200413, 20200415, ...
    date_folders = sorted(d for d in os.listdir(CLONE_DIR)
                          if d.isdigit() and len(d) == 8 and os.path.isdir(os.path.join(CLONE_DIR, d)))
    if MAX_DATE_FOLDERS is not None:
        date_folders = date_folders[:MAX_DATE_FOLDERS]
    print("Date folders to process:", len(date_folders))

    # natural sort so that 'part2' comes before 'part10'
    def natural_key(text):
        return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", text)]

    # C) extract only the cough wav files from every folder's archive
    for d in tqdm(date_folders, desc="Extracting cough audio"):
        src_folder = os.path.join(CLONE_DIR, d)
        parts = sorted([f for f in os.listdir(src_folder) if ".tar.gz" in f], key=natural_key)
        if len(parts) == 0:
            print(f"  [!] No archive found in {d} - skipped")
            continue
        dest = os.path.join(AUDIO_ROOT, d)
        os.makedirs(dest, exist_ok=True)
        # The archive may be split into several parts: join them (cat) and unpack (tar) in one go.
        cmd = ("cat " + " ".join(shlex.quote(os.path.join(src_folder, p)) for p in parts) +
               " | tar -xzf - -C " + shlex.quote(dest) +
               " --wildcards '*cough-heavy.wav' '*cough-shallow.wav'")
        res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if res.returncode != 0:
            print(f"  [!] Problem while extracting {d}: {res.stderr.strip()[:150]}")

    # D) remove the temporary clone and mark the download as finished
    shutil.rmtree(CLONE_DIR, ignore_errors=True)
    with open(DONE_MARKER, "w") as f:
        f.write("done")
    print("Download and extraction finished.")

n_wav = sum(1 for _, _, files in os.walk(AUDIO_ROOT) for f in files if f.endswith(".wav"))
print("Cough wav files available:", n_wav)

In [ ]:
# --- 1.5 Set random seeds for reproducibility --------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("Random seed set to", SEED)

## 2. Dataset Metadata Loading & Exploration

In [ ]:
# --- 2.1 Load the metadata CSV (downloaded in cell 1.4) ---------------------
meta = pd.read_csv(METADATA_CSV)
print("Metadata shape:", meta.shape)
print("Columns        :", list(meta.columns)[:15], "...")

# 'id' = participant id, 'covid_status' = health status reported in Coswara
print("\nOriginal covid_status values:")
print(meta["covid_status"].value_counts())

In [ ]:
# --- 2.2 Map the original Coswara labels into our 3 target classes ----------
#   healthy                                   -> healthy
#   positive_mild / moderate / asymp          -> covid_positive
#   recovered_full                            -> recovered
# All other statuses (e.g. 'resp_illness_not_identified', 'under_validation') are dropped.
label_map = {
    "healthy":               "healthy",
    "positive_mild":         "covid_positive",
    "positive_moderate":     "covid_positive",
    "positive_asymp":        "covid_positive",   # name used in combined_data.csv
    "positive_asymptomatic": "covid_positive",   # (kept in case the name changes)
    "recovered_full":        "recovered",
}
CLASS_NAMES = ["healthy", "covid_positive", "recovered"]   # fixed order used everywhere

meta["label"] = meta["covid_status"].map(label_map)
meta = meta.dropna(subset=["label"])
meta = meta[["id", "label"]].drop_duplicates(subset="id").rename(columns={"id": "participant_id"})
meta["participant_id"] = meta["participant_id"].astype(str)
print("Participants with a usable label:", len(meta))

In [ ]:
# --- 2.3 Find the cough audio files on disk ---------------------------------
# We walk through the audio folder and collect every cough recording.
# The parent folder name of each wav file is the participant id.
COUGH_FILES = ["cough-heavy.wav", "cough-shallow.wav"]
rows = []
for folder, _, files in os.walk(AUDIO_ROOT):
    for f in files:
        if f in COUGH_FILES:
            rows.append({
                "participant_id": os.path.basename(folder),
                "cough_type": f.replace(".wav", ""),
                "path": os.path.join(folder, f),
            })
audio_df = pd.DataFrame(rows)
print("Cough audio files found:", len(audio_df))

if len(audio_df) == 0:
    raise FileNotFoundError(
        "No cough-heavy.wav / cough-shallow.wav files found under " + AUDIO_ROOT +
        ". Re-run cell 1.4 (delete the file '.download_complete' in that folder first if needed)."
    )

In [ ]:
# --- 2.4 Merge metadata with audio files ------------------------------------
data_df = audio_df.merge(meta, on="participant_id", how="inner")
print("Recordings with both audio and label:", len(data_df))
print("Participants:", data_df["participant_id"].nunique())

# Optional: limit the number of participants per class (healthy is much bigger than the others).
# This speeds up Colab and reduces imbalance. Set to None to use everything.
MAX_PARTICIPANTS_PER_CLASS = 800

if MAX_PARTICIPANTS_PER_CLASS is not None:
    keep_ids = []
    for cls in CLASS_NAMES:
        ids = data_df.loc[data_df["label"] == cls, "participant_id"].unique()
        rng = np.random.RandomState(SEED)
        rng.shuffle(ids)
        keep_ids.extend(ids[:MAX_PARTICIPANTS_PER_CLASS])
    data_df = data_df[data_df["participant_id"].isin(keep_ids)].reset_index(drop=True)

print("\nFinal dataset (recordings per class):")
print(data_df["label"].value_counts())
print("\nFinal dataset (participants per class):")
print(data_df.groupby("label")["participant_id"].nunique())

In [ ]:
# --- 2.5 Plot the class distribution ----------------------------------------
counts = data_df["label"].value_counts().reindex(CLASS_NAMES).fillna(0)

plt.figure(figsize=(6, 4))
bars = plt.bar(counts.index, counts.values, color=["#4C9F70", "#D9534F", "#5B8DEF"])
for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             int(bar.get_height()), ha="center", va="bottom", fontsize=10)
plt.title("Class Distribution (number of cough recordings)")
plt.ylabel("Recordings")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"), dpi=150)
plt.show()

## 3. Audio Preprocessing & Feature Extraction
Steps for every recording: load at 16 kHz -> trim silence -> normalise volume -> pad/crop to 3 s -> log-Mel spectrogram.

In [ ]:
# --- 3.1 Audio / spectrogram settings ---------------------------------------
SR        = 16000          # target sampling rate (Hz)
DURATION  = 3.0            # every clip is fixed to 3 seconds
N_SAMPLES = int(SR * DURATION)
N_MELS    = 64             # number of Mel bands (spectrogram height)
N_FFT     = 1024
HOP       = 256            # -> about 188 time frames for 3 seconds

def extract_logmel(path):
    """Load one wav file and return a (N_MELS, frames) log-Mel spectrogram, or None if the file is bad."""
    try:
        audio, _ = librosa.load(path, sr=SR, mono=True)       # load + resample to 16 kHz
        if len(audio) == 0:
            return None, None
        audio, _ = librosa.effects.trim(audio, top_db=30)      # remove leading/trailing silence
        if len(audio) == 0:
            return None, None
        audio = audio / (np.max(np.abs(audio)) + 1e-9)         # normalise volume to [-1, 1]
        if len(audio) < N_SAMPLES:                             # too short -> pad with zeros
            audio = np.pad(audio, (0, N_SAMPLES - len(audio)))
        else:                                                  # too long -> crop
            audio = audio[:N_SAMPLES]
        mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_fft=N_FFT,
                                             hop_length=HOP, n_mels=N_MELS)
        log_mel = librosa.power_to_db(mel, ref=np.max)         # convert power to decibels
        return audio, log_mel
    except Exception as e:
        print("Could not read", path, "->", e)
        return None, None

In [ ]:
# --- 3.2 Visual examples: waveform + log-Mel spectrogram for each class -----
fig, axes = plt.subplots(len(CLASS_NAMES), 2, figsize=(12, 3.2 * len(CLASS_NAMES)))

for i, cls in enumerate(CLASS_NAMES):
    subset = data_df[data_df["label"] == cls]
    audio, log_mel = None, None

    # try the first few files of this class until one loads correctly
    for path in subset["path"].head(10):
        audio, log_mel = extract_logmel(path)
        if audio is not None:
            break

    if audio is None:
        axes[i, 0].set_title(f"{cls}: no valid audio found")
        continue

    # left: waveform
    t = np.arange(len(audio)) / SR
    axes[i, 0].plot(t, audio, color="steelblue", linewidth=0.7)
    axes[i, 0].set_title(f"Waveform - {cls}")
    axes[i, 0].set_xlabel("Time (s)")
    axes[i, 0].set_ylabel("Amplitude")

    # right: log-Mel spectrogram
    img = librosa.display.specshow(log_mel, sr=SR, hop_length=HOP, x_axis="time",
                                   y_axis="mel", fmax=SR // 2, ax=axes[i, 1], cmap="magma")
    axes[i, 1].set_title(f"Log-Mel Spectrogram - {cls}")
    fig.colorbar(img, ax=axes[i, 1], format="%+2.0f dB")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "audio_examples.png"), dpi=150)
plt.show()

In [ ]:
# --- 3.3 Extract features for ALL recordings --------------------------------
# This can take several minutes. Bad/corrupt files are skipped automatically.
from tqdm.auto import tqdm

features = []      # list of spectrograms
good_rows = []     # row indices that were processed successfully

for idx, path in enumerate(tqdm(data_df["path"], desc="Extracting log-Mel")):
    _, log_mel = extract_logmel(path)
    if log_mel is not None:
        features.append(log_mel)
        good_rows.append(idx)

# keep only the rows that worked, so features and labels stay aligned
n_skipped = len(data_df) - len(good_rows)
data_df = data_df.iloc[good_rows].reset_index(drop=True)
X = np.array(features, dtype=np.float32)
X = X[..., np.newaxis]                     # add channel dimension -> (N, mels, frames, 1)

print("Feature array shape:", X.shape)
print("Skipped files      :", n_skipped)
print("Recordings kept    :", len(data_df))

## 4. Patient-Level Train / Validation / Test Split
One participant can have several recordings (heavy + shallow cough). If recordings of the same person
appear in both train and test, the model can "recognise the person" instead of learning the cough -> **data leakage**.
So we split the **participants** first, and then give every recording to the split of its participant.

In [ ]:
# --- 4.1 Split participant IDs (stratified by class) ------------------------
from sklearn.model_selection import train_test_split

# one row per participant with their class label
participants = data_df.groupby("participant_id")["label"].first().reset_index()

# 70% train / 15% validation / 15% test
train_ids, temp_ids = train_test_split(
    participants["participant_id"], test_size=0.30,
    stratify=participants["label"], random_state=SEED)

temp_labels = participants.set_index("participant_id").loc[temp_ids, "label"]
val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.50, stratify=temp_labels, random_state=SEED)

# safety check: no participant may appear in two splits
assert len(set(train_ids) & set(val_ids)) == 0
assert len(set(train_ids) & set(test_ids)) == 0
assert len(set(val_ids) & set(test_ids)) == 0
print("No participant overlap between splits - leakage check passed.")

In [ ]:
# --- 4.2 Build the arrays for each split ------------------------------------
label_to_idx = {name: i for i, name in enumerate(CLASS_NAMES)}
y_all = data_df["label"].map(label_to_idx).values

train_mask = data_df["participant_id"].isin(train_ids).values
val_mask   = data_df["participant_id"].isin(val_ids).values
test_mask  = data_df["participant_id"].isin(test_ids).values

X_train, y_train = X[train_mask], y_all[train_mask]
X_val,   y_val   = X[val_mask],   y_all[val_mask]
X_test,  y_test  = X[test_mask],  y_all[test_mask]

# Normalise using statistics of the TRAIN set only (never look at validation/test data)
mean, std = X_train.mean(), X_train.std() + 1e-9
X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std
X_test  = (X_test  - mean) / std

# Show the size of each split
summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "participants": [len(train_ids), len(val_ids), len(test_ids)],
    "recordings": [len(y_train), len(y_val), len(y_test)],
})
for i, name in enumerate(CLASS_NAMES):
    summary[name] = [(y_train == i).sum(), (y_val == i).sum(), (y_test == i).sum()]
print(summary.to_string(index=False))

In [ ]:
# --- 4.3 Class weights (helps because 'healthy' is usually the biggest class) -
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight("balanced", classes=np.arange(len(CLASS_NAMES)), y=y_train)
class_weight = {i: w for i, w in enumerate(weights)}
print("Class weights:", {CLASS_NAMES[i]: round(w, 2) for i, w in class_weight.items()})

## 5. CNN Model Training

In [ ]:
# --- 5.1 Build a simple CNN --------------------------------------------------
from tensorflow.keras import layers, models

input_shape = X_train.shape[1:]      # (mels, frames, 1)

model = models.Sequential([
    layers.Input(shape=input_shape),

    # Block 1
    layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 2
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Block 3
    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    # Classifier head
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(32, activation="relu"),
    layers.Dense(len(CLASS_NAMES), activation="softmax"),   # 3 class probabilities
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

In [ ]:
# --- 5.2 Train the model -----------------------------------------------------
# EarlyStopping stops training when validation loss stops improving and restores the best weights.
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
# --- 5.3 Plot training curves and save the model ----------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()

model.save(os.path.join(OUTPUT_DIR, "aerosense_cnn.keras"))
print("Model saved to", os.path.join(OUTPUT_DIR, "aerosense_cnn.keras"))

In [ ]:
# --- 5.4 Save everything the Streamlit app needs ----------------------------
# The app must preprocess a new recording EXACTLY like the training data, so we also store
# the normalisation values (mean/std) and all audio settings next to the model.
import json

config = {
    "class_names": CLASS_NAMES,
    "sample_rate": SR,
    "duration": DURATION,
    "n_mels": N_MELS,
    "n_fft": N_FFT,
    "hop_length": HOP,
    "norm_mean": float(mean),
    "norm_std": float(std),
    "input_shape": [int(v) for v in input_shape],
    "tensorflow_version": tf.__version__,
}
config_path = os.path.join(OUTPUT_DIR, "model_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print("Saved:", config_path)
print("Saved:", os.path.join(OUTPUT_DIR, "aerosense_cnn.keras"))

# Bundle both files into ONE zip so the browser only has to download a single file
# (browsers often block the second of two automatic downloads).
import zipfile
zip_path = os.path.join(OUTPUT_DIR, "aerosense_model_bundle.zip")
with zipfile.ZipFile(zip_path, "w") as z:
    z.write(os.path.join(OUTPUT_DIR, "aerosense_cnn.keras"), "aerosense_cnn.keras")
    z.write(config_path, "model_config.json")
print("Bundle saved:", zip_path)
print("Files inside :", zipfile.ZipFile(zip_path).namelist())

# In Colab, download the bundle to your computer (it is also always available in Google Drive).
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Automatic download skipped:", e)
    print("Get the files from Google Drive instead:", OUTPUT_DIR)

## 5B. Model Improvements
The basic CNN above is a starting point. Here we try four standard ways to improve it and compare them fairly:
1. **Hand-crafted audio features** (MFCCs, spectral shape, loudness...) + classic ML models (often strong on small datasets)
2. **Data augmentation** for the CNN (time shift, frequency/time masking, noise) so it overfits less
3. **Ensemble** (average the probabilities of several models)
4. A fair comparison using **macro-F1 / balanced accuracy** on the validation set (accuracy alone hides weak classes)

All choices are made on the VALIDATION set. The test set is only used once at the end (section 6).

In [ ]:
# --- 5B.1 Extract hand-crafted audio features --------------------------------
# For each clip we compute frame-level features and summarise them with their mean and standard deviation.
def extract_handcrafted(path):
    """Return a 1-D feature vector for one cough recording, or None if the file cannot be read."""
    try:
        audio, _ = librosa.load(path, sr=SR, mono=True)
        audio, _ = librosa.effects.trim(audio, top_db=30)
        if len(audio) == 0:
            return None
        cough_seconds = min(len(audio) / SR, DURATION)            # how long the cough really lasts
        audio = audio[:N_SAMPLES]
        audio = audio / (np.max(np.abs(audio)) + 1e-9)
        if len(audio) < 4000:                                     # very short clip -> pad a little
            audio = np.pad(audio, (0, 4000 - len(audio)))

        mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=20, n_fft=N_FFT, hop_length=HOP)
        delta1 = librosa.feature.delta(mfcc)                      # how MFCCs change over time
        delta2 = librosa.feature.delta(mfcc, order=2)
        extra = [
            librosa.feature.spectral_centroid(y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP),
            librosa.feature.spectral_bandwidth(y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP),
            librosa.feature.spectral_rolloff(y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP),
            librosa.feature.spectral_flatness(y=audio, n_fft=N_FFT, hop_length=HOP),
            librosa.feature.zero_crossing_rate(audio, frame_length=N_FFT, hop_length=HOP),
            librosa.feature.rms(y=audio, frame_length=N_FFT, hop_length=HOP),
            librosa.feature.spectral_contrast(y=audio, sr=SR, n_fft=N_FFT, hop_length=HOP, n_bands=5),
        ]
        rows = np.vstack([mfcc, delta1, delta2] + extra)          # (n_features, frames)
        vector = np.concatenate([rows.mean(axis=1), rows.std(axis=1), [cough_seconds]])
        return vector.astype(np.float32)
    except Exception as e:
        print("Handcrafted features failed for", path, "->", e)
        return None

hand_list = [extract_handcrafted(path) for path in tqdm(data_df["path"], desc="Hand-crafted features")]
dim = next(len(f) for f in hand_list if f is not None)
n_failed = sum(f is None for f in hand_list)
HAND = np.array([f if f is not None else np.zeros(dim, dtype=np.float32) for f in hand_list])
HAND = np.nan_to_num(HAND)                                         # replace any NaN/inf by 0
print("Hand-crafted feature matrix:", HAND.shape, "| failed clips:", n_failed)

In [ ]:
# --- 5B.2 Classic ML models on the hand-crafted features ----------------------
# Same patient-level split (train_mask / val_mask / test_mask) as the CNN -> no leakage.
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score

H_train, H_val, H_test = HAND[train_mask], HAND[val_mask], HAND[test_mask]
scaler = StandardScaler().fit(H_train)                             # scale using TRAIN data only
Hs_train, Hs_val, Hs_test = scaler.transform(H_train), scaler.transform(H_val), scaler.transform(H_test)

candidates = {
    "Logistic Regression": LogisticRegression(max_iter=3000, C=0.05, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=3, n_jobs=-1,
                                            class_weight="balanced_subsample", random_state=SEED),
    "Gradient Boosting": HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=250,
                                                        class_weight="balanced", random_state=SEED),
}

def score_probs(y_true, probs):
    """Return (accuracy, balanced accuracy, macro-F1) for predicted probabilities."""
    pred = np.argmax(probs, axis=1)
    return ((pred == y_true).mean(),
            balanced_accuracy_score(y_true, pred),
            f1_score(y_true, pred, average="macro", zero_division=0))

classic_val_probs, classic_test_probs = {}, {}
print(f"{'model':<22}{'val acc':>9}{'val bal.acc':>13}{'val macroF1':>13}")
for name, clf in candidates.items():
    clf.fit(Hs_train, y_train)
    # make sure the probability columns always follow CLASS_NAMES order, even if a class is missing
    def full_proba(x):
        out = np.zeros((len(x), len(CLASS_NAMES)))
        out[:, clf.classes_] = clf.predict_proba(x)
        return out
    classic_val_probs[name], classic_test_probs[name] = full_proba(Hs_val), full_proba(Hs_test)
    acc, bacc, f1 = score_probs(y_val, classic_val_probs[name])
    print(f"{name:<22}{acc:>9.3f}{bacc:>13.3f}{f1:>13.3f}")

best_classic = max(classic_val_probs, key=lambda n: score_probs(y_val, classic_val_probs[n])[2])
print("\nBest classic model on validation (macro-F1):", best_classic)

In [ ]:
# --- 5B.3 CNN with data augmentation -----------------------------------------
# Augmentation = randomly changing the training spectrograms a little each epoch so the CNN
# cannot simply memorise them. It is applied to TRAINING data only (validation/test stay untouched).
F_BINS, T_FRAMES = X_train.shape[1], X_train.shape[2]

def augment(x, y, w):
    """x: one normalised log-Mel spectrogram of shape (mels, frames, 1)."""
    # 1) random time shift (roll the spectrogram left/right)
    x = tf.roll(x, shift=tf.random.uniform([], -25, 26, dtype=tf.int32), axis=1)
    # 2) SpecAugment: blank out one band of frequencies and one stretch of time
    f_width = tf.random.uniform([], 0, 9, dtype=tf.int32)
    f_start = tf.random.uniform([], 0, F_BINS - 8, dtype=tf.int32)
    t_width = tf.random.uniform([], 0, 31, dtype=tf.int32)
    t_start = tf.random.uniform([], 0, T_FRAMES - 30, dtype=tf.int32)
    freq_idx, time_idx = tf.range(F_BINS), tf.range(T_FRAMES)
    freq_mask = (freq_idx >= f_start) & (freq_idx < f_start + f_width)
    time_mask = (time_idx >= t_start) & (time_idx < t_start + t_width)
    mask = freq_mask[:, tf.newaxis] | time_mask[tf.newaxis, :]
    x = tf.where(mask[..., tf.newaxis], tf.zeros_like(x), x)          # 0 = the average value
    # 3) small random loudness change + a little noise
    x = x + tf.random.uniform([], -0.2, 0.2) + tf.random.normal(tf.shape(x), stddev=0.05)
    return x, y, w

def build_cnn():
    """Same simple CNN as section 5.1 (kept as a function so we can build a second copy)."""
    m = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(16, (3, 3), padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"), layers.BatchNormalization(), layers.MaxPooling2D((2, 2)),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(32, activation="relu"),
        layers.Dense(len(CLASS_NAMES), activation="softmax"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

# class weights become per-sample weights (rare classes count more)
cw_vector = np.array([class_weight[i] for i in range(len(CLASS_NAMES))], dtype=np.float32)
sample_w = cw_vector[y_train]

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train, sample_w))
            .shuffle(2048, seed=SEED)
            .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(32)
            .prefetch(tf.data.AUTOTUNE))

tf.keras.backend.clear_session()
cnn_aug = build_cnn()
history_aug = cnn_aug.fit(
    train_ds, validation_data=(X_val, y_val), epochs=60, verbose=1,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True),
               tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=1)])

In [ ]:
# --- 5B.4 Compare all models on the VALIDATION set and pick the best ---------
val_probs = {
    "CNN (baseline)":      model.predict(X_val, verbose=0),
    "CNN + augmentation":  cnn_aug.predict(X_val, verbose=0),
    best_classic:          classic_val_probs[best_classic],
}
test_probs = {
    "CNN (baseline)":      model.predict(X_test, verbose=0),
    "CNN + augmentation":  cnn_aug.predict(X_test, verbose=0),
    best_classic:          classic_test_probs[best_classic],
}
# Ensemble = average of the probabilities of the augmented CNN and the best classic model
ens_name = "Ensemble (CNN+aug & " + best_classic + ")"
val_probs[ens_name]  = (val_probs["CNN + augmentation"]  + val_probs[best_classic]) / 2
test_probs[ens_name] = (test_probs["CNN + augmentation"] + test_probs[best_classic]) / 2

print(f"{'model (validation set)':<46}{'acc':>7}{'bal.acc':>9}{'macroF1':>9}")
for name, probs in val_probs.items():
    acc, bacc, f1 = score_probs(y_val, probs)
    print(f"{name:<46}{acc:>7.3f}{bacc:>9.3f}{f1:>9.3f}")

# choose by validation macro-F1 (treats all 3 classes equally, unlike plain accuracy)
FINAL_MODEL_NAME = max(val_probs, key=lambda n: score_probs(y_val, val_probs[n])[2])
FINAL_PROB_TEST = test_probs[FINAL_MODEL_NAME]
print("\nSelected model:", FINAL_MODEL_NAME, "-> it will be evaluated on the test set in section 6.")

In [ ]:
# --- 5B.5 Optional: binary view "COVID-positive vs everything else" ---------
# Telling 'recovered' apart from 'healthy' by cough alone is very hard. The binary question
# "positive or not?" is more realistic and is worth reporting as an extra result.
pos_idx = label_to_idx["covid_positive"]
yb_train, yb_val, yb_test = (y_train == pos_idx), (y_val == pos_idx), (y_test == pos_idx)

if yb_test.sum() > 0 and yb_test.sum() < len(yb_test) and yb_val.sum() > 0:
    binary_clf = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=250,
                                                class_weight="balanced", random_state=SEED)
    binary_clf.fit(Hs_train, yb_train)
    score_val, score_test = binary_clf.predict_proba(Hs_val)[:, 1], binary_clf.predict_proba(Hs_test)[:, 1]

    # choose the decision threshold on the validation set (best TPR - FPR), then apply to test
    from sklearn.metrics import roc_curve
    fpr_v, tpr_v, thr_v = roc_curve(yb_val, score_val)
    threshold = thr_v[np.argmax(tpr_v - fpr_v)]
    pred_b = score_test >= threshold
    print(f"Binary (positive vs not) - test ROC-AUC     : {roc_auc_score(yb_test, score_test):.3f}")
    print(f"Binary (positive vs not) - test balanced acc: {balanced_accuracy_score(yb_test, pred_b):.3f}")
    print(f"(threshold {threshold:.2f} chosen on the validation set)")
else:
    print("Not enough positive/negative examples in validation/test for the binary view.")

In [ ]:
# --- 5B.6 Save the selected classic model for the Streamlit app ---------------
# If a classic model (e.g. Gradient Boosting) won in 5B.4, the app does NOT need TensorFlow.
# We save the model + scaler + audio settings in ONE file, so only one download is needed.
import joblib
import sklearn

if "FINAL_MODEL_NAME" in globals() and FINAL_MODEL_NAME != best_classic:
    print(f"[!] The selected model was '{FINAL_MODEL_NAME}', but this cell saves '{best_classic}'. "
          "Ask your mentor/assistant how to deploy the ensemble or CNN instead.")

bundle = {
    "model": candidates[best_classic],            # the fitted scikit-learn model
    "model_name": best_classic,
    "scaler": scaler,                             # same scaling as in training
    "class_names": CLASS_NAMES,
    "settings": {"sr": SR, "duration": DURATION, "n_fft": N_FFT, "hop": HOP},
    "n_features": int(HAND.shape[1]),
    "sklearn_version": sklearn.__version__,
}
bundle_path = os.path.join(OUTPUT_DIR, "aerosense_classic.joblib")
joblib.dump(bundle, bundle_path)
print("Saved:", bundle_path)
print("scikit-learn version used for training:", sklearn.__version__)
print(">>> Put this exact line in the app's requirements.txt:  scikit-learn==" + sklearn.__version__)

try:
    from google.colab import files
    files.download(bundle_path)                   # a single file -> the browser will not block it
except Exception as e:
    print("Automatic download skipped:", e, "-> take the file from Google Drive:", OUTPUT_DIR)

## 6. Evaluation & Advanced Metrics

In [ ]:
# --- 6.1 Predict on the test set and print basic metrics --------------------
from sklearn.metrics import classification_report, balanced_accuracy_score, f1_score

# If section 5B ran, evaluate the model selected there; otherwise evaluate the basic CNN.
if "FINAL_PROB_TEST" in globals():
    y_prob = FINAL_PROB_TEST
    print("Evaluating:", FINAL_MODEL_NAME)
else:
    y_prob = model.predict(X_test, verbose=0)       # probabilities, shape (N, 3)
y_pred = np.argmax(y_prob, axis=1)                  # predicted class index

test_acc = float((y_pred == y_test).mean())
print(f"Test accuracy         : {test_acc:.4f}")
print(f"Test balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.4f}   (mean recall over classes)")
print(f"Test macro-F1         : {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}\n")

print(classification_report(y_test, y_pred, labels=np.arange(len(CLASS_NAMES)),
                            target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# --- 6.2 Confusion matrix (counts and row-normalised) -----------------------
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(CLASS_NAMES)))
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)     # each row sums to 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title("Confusion Matrix (counts)")
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Greens", ax=axes[1], vmin=0, vmax=1,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[1].set_title("Confusion Matrix (normalised per true class)")
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

In [ ]:
# --- 6.3 Multi-class ROC-AUC curves (per class + micro + macro) -------------
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

n_classes = len(CLASS_NAMES)
y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))   # one-hot true labels

colors = ["#4C9F70", "#D9534F", "#5B8DEF"]
fpr, tpr, roc_auc = {}, {}, {}

# per-class ROC (one-vs-rest). A class needs both positive and negative samples in the test set.
valid_classes = []
for i in range(n_classes):
    if y_test_bin[:, i].sum() == 0 or y_test_bin[:, i].sum() == len(y_test_bin):
        print(f"Skipping ROC for '{CLASS_NAMES[i]}' (needs both positive and negative test samples)")
        continue
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    valid_classes.append(i)

# micro-average: pool all classes together
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_prob.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# macro-average: average the per-class curves on a shared FPR axis
all_fpr = np.unique(np.concatenate([fpr[i] for i in valid_classes]))
mean_tpr = np.zeros_like(all_fpr)
for i in valid_classes:
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= len(valid_classes)
fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

plt.figure(figsize=(7, 6))
for i in valid_classes:
    plt.plot(fpr[i], tpr[i], color=colors[i], lw=2,
             label=f"{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.3f})")
plt.plot(fpr["micro"], tpr["micro"], color="black", linestyle=":", lw=2.5,
         label=f"micro-average (AUC = {roc_auc['micro']:.3f})")
plt.plot(fpr["macro"], tpr["macro"], color="navy", linestyle="--", lw=2.5,
         label=f"macro-average (AUC = {roc_auc['macro']:.3f})")
plt.plot([0, 1], [0, 1], color="gray", linestyle="-", lw=1, label="chance")
plt.xlim([0, 1]); plt.ylim([0, 1.02])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multi-class ROC Curves (one-vs-rest)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "roc_curves.png"), dpi=150)
plt.show()

In [ ]:
# --- 6.4 Precision-Recall curves (PR-AUC / Average Precision) ---------------
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(7, 6))
ap_scores = []

for i in range(n_classes):
    if y_test_bin[:, i].sum() == 0:
        print(f"Skipping PR curve for '{CLASS_NAMES[i]}' (no positive samples in test set)")
        continue
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_prob[:, i])
    ap = average_precision_score(y_test_bin[:, i], y_prob[:, i])
    ap_scores.append(ap)
    baseline = y_test_bin[:, i].mean()        # a random classifier's precision = class prevalence
    plt.plot(recall, precision, color=colors[i], lw=2,
             label=f"{CLASS_NAMES[i]} (AP = {ap:.3f}, baseline = {baseline:.2f})")

# micro-average PR curve
precision_m, recall_m, _ = precision_recall_curve(y_test_bin.ravel(), y_prob.ravel())
ap_micro = average_precision_score(y_test_bin, y_prob, average="micro")
plt.plot(recall_m, precision_m, color="black", linestyle=":", lw=2.5,
         label=f"micro-average (AP = {ap_micro:.3f})")

print(f"Macro-average AP (PR-AUC): {np.mean(ap_scores):.3f}")
print(f"Micro-average AP (PR-AUC): {ap_micro:.3f}")

plt.xlim([0, 1]); plt.ylim([0, 1.02])
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves (one-vs-rest)")
plt.legend(loc="upper right", fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pr_curves.png"), dpi=150)
plt.show()

In [ ]:
# --- 6.5 Final summary -------------------------------------------------------
print("=" * 50)
print("AeroSense - Final Test Results")
print("=" * 50)
print(f"Test accuracy      : {test_acc:.3f}")
print(f"ROC-AUC (macro)    : {roc_auc['macro']:.3f}")
print(f"ROC-AUC (micro)    : {roc_auc['micro']:.3f}")
print(f"PR-AUC  (macro AP) : {np.mean(ap_scores):.3f}")
print(f"PR-AUC  (micro AP) : {ap_micro:.3f}")
print("All figures and the trained model were saved to:", OUTPUT_DIR)